# v4_deepsets — entrenamiento en Google Colab (GitHub-based)

El código se clona desde GitHub en cada sesión; el dataset se baja de Drive.

## Setup previo (una sola vez)
1. Subí el HDF5 a Google Drive en `MyDrive/Ipre/datasets/v2_xyz100_step50_n5000.h5`.
2. Colab → ícono 🔑 *Secrets* (panel izquierdo) → agregá `COMET_API_KEY` con tu API key de comet.com. Activá *Notebook access*.
3. *Runtime → Change runtime type → GPU* (T4 free / A100 Pro).

## Cada sesión
Corré las celdas de arriba a abajo. La sesión arranca limpia → siempre clonás repo + copiás dataset.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo desde GitHub

In [ ]:
REPO_URL = 'https://github.com/Daspony/PMDKernel.git'
BRANCH   = 'main'

import os
if os.path.isdir('PMDKernel'):
    %cd PMDKernel
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL}
    %cd PMDKernel
print('cwd:', os.getcwd())
!git log -1 --oneline

## 3. Instalar dependencias

Colab ya trae `torch` con CUDA. Solo falta el resto.

In [ ]:
!pip install -q pytorch-lightning comet-ml h5py torchmetrics

In [ ]:
import torch, pytorch_lightning as pl, comet_ml, h5py, sklearn, torchmetrics
print(f'torch        = {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'lightning    = {pl.__version__}')
print(f'comet_ml     = {comet_ml.__version__}')
print(f'h5py         = {h5py.__version__}')
print(f'torchmetrics = {torchmetrics.__version__}')
if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')

## 4. Bajar el dataset desde Drive

Monto Drive solo para copiar el H5 al directorio local del runtime. Después podés desmontar si querés.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET = 'v2_xyz100_step50_n5000.h5'
DRIVE_SRC = f'/content/drive/MyDrive/Ipre/datasets/{DATASET}'

!mkdir -p data/datasets
!cp -v {DRIVE_SRC} data/datasets/
!ls -lh data/datasets/

## 5. Credenciales Comet

Lee `COMET_API_KEY` del panel de Secrets de Colab. Si lo dejás vacío, podés desactivar el logger pasando `os.environ['COMET_OFFLINE'] = '1'`.

In [ ]:
from google.colab import userdata
import os

os.environ['COMET_API_KEY']   = userdata.get('COMET_API_KEY')
os.environ['COMET_WORKSPACE'] = 'sebasti-n-vallejos'
print('comet workspace:', os.environ['COMET_WORKSPACE'])
print('api key len:', len(os.environ['COMET_API_KEY']))

## 6. Entrenar

Editá `RUN_TAG`, `EPOCHS` si querés.

In [ ]:
H5      = f'data/datasets/{DATASET}'
RUN_TAG = 'v4_deepsets_v2_n5000_colab'
EPOCHS  = 100

!python python/train_v4.py \
    --h5 {H5} \
    --run-tag {RUN_TAG} \
    --epochs {EPOCHS} \
    --patience 20 \
    --comet-project pmdkernel \
    --num-workers 2 \
    --pin-memory

### Smoke test (3 epochs)

In [ ]:
!python python/train_v4.py \
    --h5 data/datasets/{DATASET} \
    --run-tag v4_deepsets_colab_smoke \
    --quick \
    --num-workers 2 \
    --pin-memory

## 7. Persistir checkpoint en Drive

El runtime de Colab se borra al desconectar. Copio los logs/ckpt del run a Drive para no perderlos. Comet ya tiene métricas + hparams trackeadas.

In [ ]:
LOG_SRC = f'python/Models/v4_deepsets/logs/{RUN_TAG}'
DRIVE_DST = '/content/drive/MyDrive/Ipre/runs/'

!mkdir -p {DRIVE_DST}
!cp -rv {LOG_SRC} {DRIVE_DST}
!ls -lh {DRIVE_DST}{RUN_TAG}/

## 8. Inspeccionar resultados

In [ ]:
import torch
from pathlib import Path

log_dir  = Path('python/Models/v4_deepsets/logs') / RUN_TAG
aux_path = log_dir / f'{RUN_TAG}_aux.pt'
aux = torch.load(aux_path, weights_only=False)

print('checkpoint:', aux['ckpt_path'])
print('comet url :', aux['comet_url'])
print()
for split, m in aux['metrics'].items():
    print(f"{split:5s}  rmse={m['rmse_mt']:.4f} mT  r2={m['r2']:.4f}")